Dependencias:

In [ ]:
pip install torchaudio librosa numpy soundfile

In [ ]:
pip install sox

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for sox: filename=sox-1.5.0-py3-none-any.whl size=40036 sha256=7bf0b8f116c0ff19de0676a55616569270ff15125b049ee362563b8709610664
  Stored in directory: /root/.cache/pip/wheels/8c/c7/e7/baea1f7e79b9eb53addc81cc9b827424f4a7d8c9cc18c03659
Successfully built sox


In [ ]:
!git clone https://github.com/JorisCos/LibriMix
%cd LibriMix
!chmod +x ./generate_librimix.sh
!./generate_librimix.sh \
    mini_librimix \
    --n_src 2 \
    --freq 16k \
    --n_examples 25 \
    --duration 4 \
    --mode min


Cloning into 'LibriMix'...
remote: Enumerating objects: 274, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 274 (delta 71), reused 58 (delta 58), pack-reused 163 (from 1)
Receiving objects: 100% (274/274), 33.43 MiB | 16.72 MiB/s, done.
Resolving deltas: 100% (116/116), done.
/content/LibriMix
Download LibriSpeech/train-clean-100 into mini_librimix
Download wham_noise into mini_librimix
Download LibriSpeech/train-clean-360 into mini_librimix
Download LibriSpeech/test-clean into mini_librimix
Download LibriSpeech/dev-clean into mini_librimix
--2025-08-21 13:23:16--  http://www.openslr.org/resources/12/test-clean.tar.gz
Resolving www.openslr.org (www.openslr.org)... --2025-08-21 13:23:16--  http://www.openslr.org/resources/12/train-clean-360.tar.gz
--2025-08-21 13:23:16--  http://www.openslr.org/resources/12/dev-clean.tar.gz
Resolving www.openslr.org (www.openslr.org)... Resolving www.openslr.org (www.openslr.org)... -

In [ ]:
!apt-get -y install sox ffmpeg
%pip -q install torch torchaudio librosa soundfile numpy tqdm

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
The following additional packages will be installed:
  libopencore-amrnb0 libopencore-amrwb0 libsox-fmt-alsa libsox-fmt-base
  libsox3 libwavpack1
Suggested packages:
  libsox-fmt-all
The following NEW packages will be installed:
  libopencore-amrnb0 libopencore-amrwb0 libsox-fmt-alsa libsox-fmt-base
  libsox3 libwavpack1 sox
0 upgraded, 7 newly installed, 0 to remove and 35 not upgraded.
Need to get 617 kB of archives.
After this operation, 1,764 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libopencore-amrnb0 amd64 0.1.5-1 [94.8 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libopencore-amrwb0 amd64 0.1.5-1 [49.1 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 libsox3 amd64 14.4.2+git20190427-2+deb11u2ubuntu0.22.04.1 [240 kB]
Get

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/mini_librimix"

Gera o mini dataset com ruídos adicionados de maneira aleatória:

In [ ]:
import os, math, random, shutil
import numpy as np
import torch
import torchaudio
import soundfile as sf
from tqdm import trange

# ===== Config =====
SR = 16000          # sample rate
DUR = 4.0           # segundos por amostra
N_EX = 120          # quantas misturas gerar (ajuste conforme o espaço)
OUT_BASE = BASE_DIR
SPLIT = "train-100" # só para manter compatibilidade de pastas

# ===== Baixar um dataset minúsculo de fala =====
YESNO_DATASET_PATH = "/content/data_yesno"
os.makedirs(YESNO_DATASET_PATH, exist_ok=True)
ds = torchaudio.datasets.YESNO(YESNO_DATASET_PATH, download=True)


# Coletar caminhos prontos ou buffers de fala
def load_waveform(item_idx):
    wav, sr, _ = ds[item_idx] # Corrected unpacking
    if sr != SR:
        wav = torchaudio.functional.resample(wav, sr, SR)
    wav = wav.mean(0, keepdim=True)  # mono
    return wav

# ===== Geradores de ruído =====
def white_noise(n):
    return torch.randn(1, n)

def brown_noise(n):
    # integral do branco => espectro ~ 1/f^2
    wn = torch.randn(n)
    bn = torch.cumsum(wn, dim=0)
    bn = (bn - bn.mean()) / (bn.std() + 1e-8)
    return bn.view(1, -1)

def pink_noise(n):
    # filtro 1/f aproximado
    # gera em freq e aplica 1/sqrt(f)
    nfft = 1
    while nfft < n: nfft *= 2
    # espectro aleatório
    real = torch.randn(nfft//2+1)
    imag = torch.randn(nfft//2+1)
    spec = torch.complex(real, imag)
    freqs = torch.linspace(1, nfft//2, steps=nfft//2)  # evita f=0
    amp = torch.cat([torch.tensor([1.0]), 1.0/torch.sqrt(freqs.float())])
    spec = spec * amp
    # iFFT
    full_spec = torch.zeros(nfft, dtype=torch.complex64)
    full_spec[:nfft//2+1] = spec
    full_spec[nfft//2+1:] = torch.conj(torch.flip(spec[1:-1], dims=[0]))
    sig = torch.fft.ifft(full_spec).real
    sig = sig[:n]
    sig = (sig - sig.mean()) / (sig.std() + 1e-8)
    return sig.view(1, -1)

def hum_noise(n, sr=SR, base=60.0):
    t = torch.arange(n)/sr
    sig = (torch.sin(2*math.pi*base*t)
           + 0.5*torch.sin(2*math.pi*2*base*t)
           + 0.25*torch.sin(2*math.pi*3*base*t))
    sig = (sig - sig.mean()) / (sig.std() + 1e-8)
    return sig.view(1, -1)

NOISE_FUNCS = [white_noise, pink_noise, brown_noise, hum_noise]

# ===== Utilidades =====
def fix_length(x, target_len):
    if x.shape[1] == target_len:
        return x
    if x.shape[1] > target_len:
        return x[:, :target_len]
    # pad ou loop para completar
    reps = (target_len + x.shape[1] - 1)//x.shape[1]
    x_rep = x.repeat(1, reps)[:, :target_len]
    return x_rep

def mix_at_snr(s, n, snr_db):
    # normaliza RMS e aplica SNR
    s_rms = s.pow(2).mean().sqrt() + 1e-8
    n_rms = n.pow(2).mean().sqrt() + 1e-8
    desired_n_rms = s_rms / (10**(snr_db/20))
    n_scaled = n * (desired_n_rms / n_rms)
    mix = s + n_scaled
    # pequena normalização para evitar clip ao salvar
    peak = max(mix.abs().max().item(), 1.0)
    return (mix/peak, s/peak, n_scaled/peak)

# ===== Criar pastas =====
mix_dir = os.path.join(OUT_BASE, SPLIT, "mix_both")
s1_dir  = os.path.join(OUT_BASE, SPLIT, "s1")
noise_dir = os.path.join(OUT_BASE, SPLIT, "noise")
for d in [mix_dir, s1_dir, noise_dir]:
    os.makedirs(d, exist_ok=True)

# ===== Gerar amostras =====
target_len = int(SR*DUR)
num_items = len(ds)

rng = random.Random(1234)
for i in trange(N_EX, desc="Gerando mini_librimix"):
    idx = rng.randrange(num_items)
    wav = load_waveform(idx)          # fala
    wav = fix_length(wav, target_len)

    nf = rng.choice(NOISE_FUNCS)
    noise = nf(target_len)
    noise = fix_length(noise, target_len)

    snr = rng.uniform(0, 10)          # SNR entre 0 e 10 dB
    mix, clean, ns = mix_at_snr(wav, noise, snr)

    base = f"ex_{i:05d}.wav"
    sf.write(os.path.join(mix_dir,  base), mix.squeeze(0).numpy(), SR)
    sf.write(os.path.join(s1_dir,   base), clean.squeeze(0).numpy(), SR)
    sf.write(os.path.join(noise_dir,base), ns.squeeze(0).numpy(), SR)

print("✅ Mini-dataset criado em:", OUT_BASE)

In [ ]:
import os, random, math
import torch
import torchaudio
import soundfile as sf

# === Configurações ===
SR = 16000            # taxa de amostragem
TARGET_LEN = 6        # segundos (ajuste)
SNR_RANGE = (0, 10)   # faixa de SNR em dB
SPLIT = "train-100" # só para manter compatibilidade de pastas

# ===== Criar pastas =====
base_dir = "/content/drive/MyDrive/real_mini_librimix"
s1_dir = os.path.join(base_dir, "s1")
mix_dir = os.path.join(base_dir, "mix_both")
noise_dir = os.path.join(base_dir, "noise")

os.makedirs(mix_dir, exist_ok=True)
os.makedirs(s1_dir, exist_ok=True)
os.makedirs(noise_dir, exist_ok=True)

# === Funções utilitárias ===
def fix_length(wav, length):
    if wav.shape[1] > length:
        return wav[:, :length]
    elif wav.shape[1] < length:
        reps = (length + wav.shape[1]-1)//wav.shape[1]
        wav = wav.repeat(1, reps)[:, :length]
    return wav

# ===== Geradores de ruído =====
def white_noise(n):
    return torch.randn(1, n)

def brown_noise(n):
    # integral do branco => espectro ~ 1/f^2
    wn = torch.randn(n)
    bn = torch.cumsum(wn, dim=0)
    bn = (bn - bn.mean()) / (bn.std() + 1e-8)
    return bn.view(1, -1)

def pink_noise(n):
    # filtro 1/f aproximado
    # gera em freq e aplica 1/sqrt(f)
    nfft = 1
    while nfft < n: nfft *= 2
    # espectro aleatório
    real = torch.randn(nfft//2+1)
    imag = torch.randn(nfft//2+1)
    spec = torch.complex(real, imag)
    freqs = torch.linspace(1, nfft//2, steps=nfft//2)  # evita f=0
    amp = torch.cat([torch.tensor([1.0]), 1.0/torch.sqrt(freqs.float())])
    spec = spec * amp
    # iFFT
    full_spec = torch.zeros(nfft, dtype=torch.complex64)
    full_spec[:nfft//2+1] = spec
    full_spec[nfft//2+1:] = torch.conj(torch.flip(spec[1:-1], dims=[0]))
    sig = torch.fft.ifft(full_spec).real
    sig = sig[:n]
    sig = (sig - sig.mean()) / (sig.std() + 1e-8)
    return sig.view(1, -1)

def hum_noise(n, sr=SR, base=60.0):
    t = torch.arange(n)/sr
    sig = (torch.sin(2*math.pi*base*t)
           + 0.5*torch.sin(2*math.pi*2*base*t)
           + 0.25*torch.sin(2*math.pi*3*base*t))
    sig = (sig - sig.mean()) / (sig.std() + 1e-8)
    return sig.view(1, -1)

NOISE_FUNCS = [white_noise, pink_noise, brown_noise, hum_noise]

def mix_at_snr(s, n, snr_db):
    # normaliza RMS e aplica SNR
    s_rms = s.pow(2).mean().sqrt() + 1e-8
    n_rms = n.pow(2).mean().sqrt() + 1e-8
    desired_n_rms = s_rms / (10**(snr_db/20))
    n_scaled = n * (desired_n_rms / n_rms)
    mix = s + n_scaled
    # pequena normalização para evitar clip ao salvar
    peak = max(mix.abs().max().item(), 1.0)
    return (mix/peak, s/peak, n_scaled/peak)


# === Loop pelos áudios da pasta s1 ===
files = [f for f in os.listdir(s1_dir) if f.endswith(".flac") or f.endswith(".wav")]

for fname in files:
    wav, sr = torchaudio.load(os.path.join(s1_dir, fname))
    if sr != SR:
        wav = torchaudio.functional.resample(wav, sr, SR)
    wav = wav.mean(dim=0, keepdim=True)  # mono

    length = int(SR*TARGET_LEN)
    clean = fix_length(wav, length)

    # gera ruído
    noise = random.choice(NOISE_FUNCS)(length)
    snr = random.uniform(*SNR_RANGE)

    mix, clean_adj, noise_adj = mix_at_snr(clean, noise, snr)

    outname = os.path.splitext(fname)[0] + ".wav"
    sf.write(os.path.join(mix_dir, outname), mix.squeeze(0).numpy(), SR)
    sf.write(os.path.join(noise_dir, outname), noise_adj.squeeze(0).numpy(), SR)
    print("Feito!")
    # o clean ajustado já está em s1, então não precisa sobrescrever


Converter arquivos para .wav:

In [ ]:
import os
import torchaudio

# Caminho da pasta original com .flac
s1_flac = "/content/drive/MyDrive/real_mini_librimix/train-100/s1"
# Nova pasta para salvar .wav
s1_wav = "/content/drive/MyDrive/real_mini_librimix/train-100/s1_wav"
os.makedirs(s1_wav, exist_ok=True)

TARGET_SR = 16000  # taxa de amostragem padrão do LibriMix

for fname in os.listdir(s1_flac):
    if fname.endswith(".flac"):
        in_path = os.path.join(s1_flac, fname)
        out_path = os.path.join(s1_wav, fname.replace(".flac", ".wav"))

        # carregar
        wav, sr = torchaudio.load(in_path)

        # converter para 16kHz mono (igual usado no treino)
        if sr != TARGET_SR:
            wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
        wav = wav.mean(dim=0, keepdim=True)  # mono

        # salvar em wav
        torchaudio.save(out_path, wav, TARGET_SR)

print("✅ Conversão concluída! Arquivos salvos em:", s1_wav)


/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:337: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.save_with_torchcodec` under the hood. Some parameters like format, encoding, bits_per_sample, buffer_size, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's encoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.encoders.AudioEncoder
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:247: UserWarning: torio.io._streaming_media_encoder.StreamingMediaEncoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  s 

✅ Conversão concluída! Arquivos salvos em: /content/drive/MyDrive/real_mini_librimix/train-100/s1_wav


Montar o drive e identificar os arquivos:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

base = "/LibriMix/mini_librimix"
for root, dirs, files in os.walk(base):
    print("📂", root, "→", len(files), "arquivos")


CNN simples tipo U-Net:

In [ ]:
import os
import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import librosa
import numpy as np

### 📁 Dataset Loader
class LibriMixDataset(Dataset):
    def __init__(self, mix_dir, clean_dir, sr=16000, max_len=4):
        self.mix_dir = mix_dir
        self.clean_dir = clean_dir
        self.file_names = sorted(os.listdir(mix_dir))
        self.sr = sr
        self.max_len = max_len

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        mix_path = os.path.join(self.mix_dir, self.file_names[idx])
        clean_path = os.path.join(self.clean_dir, self.file_names[idx])

        mix, _ = torchaudio.load(mix_path)
        clean, _ = torchaudio.load(clean_path)

        # Truncar ou padronizar para tamanho fixo
        length = self.sr * self.max_len
        mix = self._fix_length(mix, length)
        clean = self._fix_length(clean, length)

        # Espectrograma
        mix_spec = self._to_spec(mix)
        clean_spec = self._to_spec(clean)

        return mix_spec, clean_spec

    def _fix_length(self, audio, length):
        if audio.shape[1] > length:
            return audio[:, :length]
        elif audio.shape[1] < length:
            pad = torch.zeros((1, length - audio.shape[1]))
            return torch.cat([audio, pad], dim=1)
        return audio

    def _to_spec(self, audio):
        spec = torch.stft(audio[0], n_fft=512, hop_length=128, return_complex=True)
        return torch.view_as_real(spec).permute(2, 0, 1)  # [2, freq, time]

### 🧠 Rede Neural (CNN simples tipo U-Net)
class DenoisingCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(2, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(32, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 2, 3, padding=1)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

### ⚙️ Treinamento
def train(model, dataloader, epochs=300, lr=1e-3):
    optim = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for mix, clean in dataloader:
            mix, clean = mix.cuda(), clean.cuda()
            out = model(mix)
            loss = loss_fn(out, clean)
            optim.zero_grad()
            loss.backward()
            optim.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

### 🔁 Inverter espectrograma para salvar áudio
def spectrogram_to_audio(spec):
    complex_spec = torch.complex(spec[0], spec[1])
    audio = torch.istft(complex_spec, n_fft=512, hop_length=128)
    return audio

### 🧪 Exemplo de uso
if __name__ == "__main__":
    mix_path = "/content/drive/MyDrive/real_mini_librimix/train-100/mix_both"
    clean_path = "/content/drive/MyDrive/real_mini_librimix/train-100/s1_wav"

    dataset = LibriMixDataset(mix_path, clean_path)
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

    model = DenoisingCNN().cuda()
    train(model, dataloader)

    # Teste com 1 amostra
    model.eval()
    mix, clean = dataset[0]
    with torch.no_grad():
        out = model(mix.unsqueeze(0).cuda())
        output_audio = spectrogram_to_audio(out[0].cpu())

    # Salvar áudio
    torchaudio.save("output_clean.wav", output_audio.unsqueeze(0), 16000)


/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

Epoch 1, Loss: 21.9223
Epoch 2, Loss: 10.4132
Epoch 3, Loss: 9.2807
Epoch 4, Loss: 8.8745
Epoch 5, Loss: 8.2693
Epoch 6, Loss: 7.8197
Epoch 7, Loss: 7.8642
Epoch 8, Loss: 7.5333
Epoch 9, Loss: 7.1285
Epoch 10, Loss: 7.1919
Epoch 11, Loss: 7.0229
Epoch 12, Loss: 6.9077
Epoch 13, Loss: 6.7374
Epoch 14, Loss: 6.7161
Epoch 15, Loss: 6.7026
Epoch 16, Loss: 6.6021
Epoch 17, Loss: 6.6300
Epoch 18, Loss: 6.5344
Epoch 19, Loss: 6.3939
Epoch 20, Loss: 6.3072
Epoch 21, Loss: 6.2543
Epoch 22, Loss: 6.2720
Epoch 23, Loss: 6.2473
Epoch 24, Loss: 6.1730
Epoch 25, Loss: 6.1421
Epoch 26, Loss: 6.1938
Epoch 27, Loss: 6.0732
Epoch 28, Loss: 5.9713
Epoch 29, Loss: 6.4817
Epoch 30, Loss: 6.1713
Epoch 31, Loss: 5.8935
Epoch 32, Loss: 6.3749
Epoch 33, Loss: 6.0056
Epoch 34, Loss: 5.8127
Epoch 35, Loss: 6.2825
Epoch 36, Loss: 5.9521
Epoch 37, Loss: 6.3553
Epoch 38, Loss: 5.8307
Epoch 39, Loss: 5.7882
Epoch 40, Loss: 5.7436
Epoch 41, Loss: 5.7297
Epoch 42, Loss: 5.9434
Epoch 43, Loss: 5.9989
Epoch 44, Loss: 5.

/tmp/ipython-input-2764093760.py:94: UserWarning: A window was not provided. A rectangular window will be applied.Please provide the same window used by stft to make the inversion lossless.To suppress this warning and use a rectangular window, explicitly set `window=torch.ones(n_fft, device=<device>)`. (Triggered internally at /pytorch/aten/src/ATen/native/SpectralOps.cpp:1036.)
  audio = torch.istft(complex_spec, n_fft=512, hop_length=128)
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:337: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.save_with_torchcodec` under the hood. Some parameters like format, encoding, bits_per_sample, buffer_size, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's encoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.encoders.AudioEncoder
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpe

Salva o modelo na pasta corrente como denoiser_model.pht:

In [ ]:
torch.save(model.state_dict(), "denoiser_model.pth")

AttributeError: 'str' object has no attribute 'state_dict'

Calcula o SDR e o MSE para todos os audios:

In [ ]:
#import os
import torch
import librosa
import numpy as np
import soundfile as sf

# -----------------------------
# Funções utilitárias
# -----------------------------
def load_audio(file_path):
    y, sr = librosa.load(file_path, sr=None)
    return y, sr

def save_audio(y, sr, path):
    sf.write(path, y, sr)

def mse_loss(clean, enhanced):
    clean_t = torch.tensor(clean, dtype=torch.float32)
    enh_t = torch.tensor(enhanced, dtype=torch.float32)
    min_len = min(len(clean_t), len(enh_t))
    clean_t = clean_t[:min_len]
    enh_t = enh_t[:min_len]
    return torch.mean((clean_t - enh_t) ** 2).item()

def sdr_metric(clean, enhanced, eps=1e-8):
    min_len = min(len(clean), len(enhanced))
    clean = clean[:min_len]
    enhanced = enhanced[:min_len]
    num = np.sum(clean ** 2)
    den = np.sum((clean - enhanced) ** 2) + eps
    return 10 * np.log10(num / den)

def waveform_to_spectrogram(audio):
    # Assuming audio is a 1D numpy array
    audio_tensor = torch.tensor(audio, dtype=torch.float32).unsqueeze(0) # Add batch dimension
    spec = torch.stft(audio_tensor[0], n_fft=512, hop_length=128, return_complex=True)
    return torch.view_as_real(spec).permute(2, 0, 1)  # [2, freq, time]

def spectrogram_to_audio(spec):
    # Assuming spec is a 3D tensor [2, freq, time]
    complex_spec = torch.complex(spec[0], spec[1])
    audio = torch.istft(complex_spec, n_fft=512, hop_length=128)
    return audio


# -----------------------------
# Testando a rede em todos os áudios
# -----------------------------
def test_model_on_dataset(model, mix_dir, clean_dir, output_dir, device="cuda"):
    os.makedirs(output_dir, exist_ok=True)

    mse_results = []
    sdr_results = []

    files = [f for f in os.listdir(mix_dir) if f.endswith(".wav")]

    for fname in files:
        mix_path = os.path.join(mix_dir, fname)
        clean_path = os.path.join(clean_dir, fname)

        if not os.path.exists(clean_path):
            print(f"⚠️ Clean não encontrado para {fname}, pulando...")
            continue

        # carregar mix e clean
        mix, sr = load_audio(mix_path)
        clean, _ = load_audio(clean_path)

        # preparar entrada para rede (converter para espectrograma)
        mix_spec = waveform_to_spectrogram(mix).unsqueeze(0).to(device) # Add batch dimension

        # inferência
        with torch.no_grad():
            enhanced_spec = model(mix_spec)

        # converter de volta para áudio
        enhanced_audio = spectrogram_to_audio(enhanced_spec.squeeze(0).cpu())

        # salvar resultado
        outname = os.path.splitext(fname)[0] + "_IA.wav"
        save_audio(enhanced_audio.numpy(), sr, os.path.join(output_dir, outname))

        # métricas
        mse_val = mse_loss(clean, enhanced_audio.numpy())
        sdr_val = sdr_metric(clean, enhanced_audio.numpy())

        mse_results.append(mse_val)
        sdr_results.append(sdr_val)

        print(f"{fname} -> MSE: {mse_val:.6f} | SDR: {sdr_val:.2f} dB")

    print("\n=== RESULTADOS MÉDIOS ===")
    print(f"MSE médio: {np.mean(mse_results):.6f}")
    print(f"SDR médio: {np.mean(sdr_results):.2f} dB")

Testa o modelo com os datasets de maneira mais organizada mostrando o SDR e MSE no final:

In [ ]:
mix_dir = "/content/drive/MyDrive/real_mini_librimix/mix_both"
clean_dir = "/content/drive/MyDrive/real_mini_librimix/s1"
output_dir = "/content/drive/MyDrive/real_mini_librimix/IA_outputs"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = DenoisingCNN().cuda()
model.load_state_dict(torch.load("denoiser_model.pth"))
model.to(device)
model.eval()

test_model_on_dataset(model, mix_dir, clean_dir, output_dir, device)

19-198-0003.wav -> MSE: 0.000103 | SDR: 14.82 dB
19-198-0008.wav -> MSE: 0.000081 | SDR: 15.22 dB
19-198-0023.wav -> MSE: 0.000716 | SDR: 7.18 dB
19-198-0037.wav -> MSE: 0.000088 | SDR: 15.52 dB
19-198-0031.wav -> MSE: 0.000166 | SDR: 15.13 dB
19-198-0013.wav -> MSE: 0.000589 | SDR: 7.08 dB
19-198-0000.wav -> MSE: 0.000077 | SDR: 14.57 dB
19-198-0002.wav -> MSE: 0.000122 | SDR: 15.13 dB
19-198-0016.wav -> MSE: 0.000267 | SDR: 11.07 dB
19-198-0036.wav -> MSE: 0.000402 | SDR: 10.16 dB
19-198-0022.wav -> MSE: 0.000295 | SDR: 11.40 dB
19-198-0006.wav -> MSE: 0.000307 | SDR: 11.76 dB
19-198-0028.wav -> MSE: 0.000075 | SDR: 14.40 dB
19-198-0019.wav -> MSE: 0.000138 | SDR: 15.44 dB
19-198-0026.wav -> MSE: 0.000133 | SDR: 14.93 dB
19-198-0032.wav -> MSE: 0.000096 | SDR: 15.68 dB
19-198-0034.wav -> MSE: 0.001253 | SDR: 6.68 dB
19-198-0021.wav -> MSE: 0.000077 | SDR: 14.91 dB
19-198-0027.wav -> MSE: 0.000097 | SDR: 16.61 dB
19-198-0009.wav -> MSE: 0.000074 | SDR: 14.64 dB
19-198-0014.wav -> MSE:

Carrega um arquivo do computador:

In [ ]:
from google.colab import files

uploaded = files.upload()  # escolha um arquivo .wav do seu PC
audio_path = list(uploaded.keys())[0]
print("Arquivo carregado:", audio_path)

Saving input_audio.wav to input_audio.wav
Arquivo carregado: input_audio.wav


Mostra características do modelo:

In [ ]:
import torch

model = DenoisingCNN().cuda()
model.load_state_dict(torch.load("denoiser_model.pth"))
model.eval()


DenoisingCNN(
  (encoder): Sequential(
    (0): Conv2d(2, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
  )
  (decoder): Sequential(
    (0): Conv2d(32, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(16, 2, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  )
)

Testes com outros modelos, a incrementar:

In [ ]:
import torchaudio

# carregar áudio externo
wav, sr = torchaudio.load(audio_path)

# converter para 16kHz e mono (igual ao treino)
if sr != 16000:
    wav = torchaudio.functional.resample(wav, sr, 16000)
wav = wav.mean(dim=0, keepdim=True)  # mono

# STFT
spec = torch.stft(wav[0], n_fft=512, hop_length=128, return_complex=True)
spec_ri = torch.stack([spec.real, spec.imag], dim=0).unsqueeze(0).cuda()

# passar pelo modelo
with torch.no_grad():
    out = model(spec_ri).cpu().squeeze(0)

# ISTFT
out_complex = torch.complex(out[0], out[1])
denoised = torch.istft(out_complex, n_fft=512, hop_length=128)

# salvar saída
torchaudio.save("audio_filtrado.wav", denoised.unsqueeze(0), 16000)
print("✅ Áudio filtrado salvo como audio_filtrado.wav")

✅ Áudio filtrado salvo como audio_filtrado.wav


Visualizaçao de resultados:

In [ ]:
import IPython.display as ipd

print("➡️ Original:")
ipd.Audio(audio_path)



➡️ Original:


In [ ]:
print("🎯 Filtrado:")
ipd.Audio("audio_filtrado.wav")

🎯 Filtrado:


In [ ]:
import IPython.display as ipd

#print("➡️ Áudio com ruído:")
#ipd.Audio("mix.wav")

#print("✅ Áudio limpo (ground truth):")
#ipd.Audio("clean.wav")

print("🎯 Áudio processado pela RNA:")
ipd.Audio("output_clean.wav")


🎯 Áudio processado pela RNA:
